# Patent Landscaping — Snorkel vs MAS (Colab GPU)

**Just refresh → Runtime: GPU (T4) → Run all, top to bottom.** It is restart-safe:
- data prep + Snorkel labeling are fast and always run;
- each SciBERT model is **restored from Google Drive if present, otherwise trained (~30 min) and saved to Drive**;
- then train/val curves, threshold calibration, and the **recall ablation (in-domain negatives)**.

First run trains once (~1 h); later sessions restore in seconds. MAS labels (`DataSet/mas/mas_ranked_scores.csv`) ship with the clone.

In [ ]:
# 1) setup — clone (once) + update + install.  Absolute paths avoid nested clones.
import os
%cd /content
REPO = 'https://github.com/PassionChicken-Leesuin/Patent_Landscaping_Final.git'
if not os.path.exists('/content/Patent_Landscaping_Final'):
    !git clone $REPO
%cd /content/Patent_Landscaping_Final
!git pull
!pip -q install snorkel transformers datasets scikit-learn accelerate

In [ ]:
# 2) mount Google Drive (persistent store for trained models)
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/patent_landscaping_outputs'
os.makedirs(DRIVE, exist_ok=True)
os.makedirs('outputs', exist_ok=True)
print('model store on Drive:', DRIVE)

In [ ]:
# 3) data prep (fast, always run) — dedup + negative pool + Snorkel labels
!python -m scripts.build_dataset
!python -m scripts.run_snorkel

## 4) Get each model: local → Drive → train, keeping Drive in sync
`FORCE=True` retrains from scratch. `ensure(name, args)` is reused for the ablation below.

In [ ]:
import shutil
FORCE = False
EPOCHS, MAX_LEN = 4, 256

def ensure(name, run_args):
    """name = output tag (e.g. 'mas', 'mas_equalN'); run_args = run_downstream flags."""
    local, dstore = f'outputs/scibert_{name}', f'{DRIVE}/scibert_{name}'
    if not FORCE and os.path.isfile(f'{local}/config.json'):
        if not os.path.isfile(f'{dstore}/config.json'):
            shutil.copytree(local, dstore, dirs_exist_ok=True)
            if os.path.isfile(f'outputs/metrics_{name}.json'): shutil.copy(f'outputs/metrics_{name}.json', DRIVE)
        print(f'[{name}] local'); return
    if not FORCE and os.path.isfile(f'{dstore}/config.json'):
        shutil.copytree(dstore, local, dirs_exist_ok=True)
        if os.path.isfile(f'{DRIVE}/metrics_{name}.json'): shutil.copy(f'{DRIVE}/metrics_{name}.json', 'outputs/')
        print(f'[{name}] restored from Drive'); return
    print(f'[{name}] training (~30 min) ...')
    get_ipython().system(f'python -m scripts.run_downstream {run_args} --epochs {EPOCHS} --max-len {MAX_LEN}')
    shutil.copytree(local, dstore, dirs_exist_ok=True)
    if os.path.isfile(f'outputs/metrics_{name}.json'): shutil.copy(f'outputs/metrics_{name}.json', DRIVE)
    print(f'[{name}] done + saved to Drive')

# baseline (all OOD negatives)
ensure('snorkel', '--arm snorkel')
ensure('mas',     '--arm mas')

In [ ]:
# 5) training/validation curves (baseline Fig 3/4 style)
from src.downstream.plots import plot_history
for arm in ['snorkel', 'mas']:
    try:
        plot_history(arm)
    except FileNotFoundError:
        print(f'[{arm}] no history.json — skip')

## 6) Threshold calibration + ROC/PR
Tunes the threshold on validation (never gold) and re-reports on gold, with ROC/PR + score histograms.

In [ ]:
!python -m scripts.calibrate_eval

## 7) Recall ablation — in-domain negatives (equal-N)
Diagnosis: at 0.5 both models predict almost all-negative because the **out-of-domain negatives are too easy** — the model learns 'vehicle-vocab vs blockchain-vocab' instead of the real boundary (autonomous-driving SEED vs autonomous-driving look-alike), so recall on real SEED collapses.

This ablation cuts the OOD negatives down to **equal-N (negatives ≈ positives, in-domain hard negatives first)**, forcing the model to learn the true boundary. If the hypothesis holds, **recall jumps** versus the baseline. Trains 2 more models (~30 min each, saved to Drive).

In [ ]:
ensure('mas_equalN',     '--arm mas --neg-pos-ratio 1.0 --tag equalN')
ensure('snorkel_equalN', '--arm snorkel --neg-pos-ratio 1.0 --tag equalN')

In [ ]:
# 7b) compare baseline vs equal-N (gold eval @0.5)
import json
def show(name):
    p = f'outputs/metrics_{name}.json'
    if not os.path.isfile(p):
        print(f'{name:16s} (missing)'); return
    r = json.load(open(p))
    seed = r['by_expansion_level'].get('SEED', {}).get('recall(TP rate)')
    print(f"{name:16s} F1={r['macro_f1']:.3f} AUC={r['auc']:.3f} "
          f"AP={r.get('average_precision', float('nan')):.3f} R={r['recall']:.3f} "
          f"P={r['precision']:.3f} | SEED-recall={seed:.3f}")
for n in ['snorkel', 'snorkel_equalN', 'mas', 'mas_equalN']:
    show(n)